# try kubakub nodes in your browser

ComfyUI + [kubakub nodes](https://github.com/kubakubkub/ComfyUI_KubaNodes) on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**
2. Run the three cells from top to bottom (**Runtime → Run all**), about 10 minutes with the models.
3. The last cell prints a link: open it, then **Workflow → Browse templates → ComfyUI_KubaNodes**.

Without any model these run at once: **01 regions in 30 seconds**, **03 director playground**, **shader**,
**mask_animate**, **sound**, **region_materials** and the **kubakub timeline** (bottom panel).
With the models (cell 2): **02 one prompt per region**, a facade made region by region with Flux 2 Klein 4B.
Every file field left empty runs a built-in sample facade, so nothing has to be uploaded.

In [ ]:
# 1. ComfyUI 0.43.0 + the newest kubakub nodes (about 2 minutes)
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!git clone -q --depth 1 --branch v0.43.0 https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!git clone -q --depth 1 --branch main https://github.com/kubakubkub/ComfyUI_KubaNodes custom_nodes/ComfyUI_KubaNodes
!pip install -q -r custom_nodes/ComfyUI_KubaNodes/requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

**2. Flux 2 Klein 4B** for workflow 02 (about 10 GB of downloads, 3-8 minutes).
The fp8 files fit Colab's memory; they are saved under the names the workflows load.
Skip this cell to try the nodes without AI models first.

In [ ]:
# 2. models (optional)
%cd /content/ComfyUI/models
!wget -q -c -O diffusion_models/flux-2-klein-4b.safetensors https://huggingface.co/black-forest-labs/FLUX.2-klein-4b-fp8/resolve/main/flux-2-klein-4b-fp8.safetensors
!wget -q -c -O text_encoders/qwen_3_4b.safetensors https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors
!wget -q -c -O vae/flux2-vae.safetensors https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors
!find . -name '*.safetensors' -size -1M -print -delete  # a failed download, so a re-run fetches it again
!ls -la diffusion_models text_encoders vae | grep safetensors
%cd /content/ComfyUI

In [ ]:
# 3. start ComfyUI and open a public link to it (cloudflared quick tunnel, no account); safe to run again
!pkill -f "[m]ain.py --listen" ; pkill cloudflared ; true
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null
import subprocess, time, re, socket
server = subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"], cwd="/content/ComfyUI",
                          stdout=open("/content/comfy.log", "w"), stderr=subprocess.STDOUT)
for _ in range(300):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI stopped while starting:\n" + open("/content/comfy.log").read()[-3000:])
    try:
        socket.create_connection(("127.0.0.1", 8188), 1).close(); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not start in 5 minutes, see /content/comfy.log")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    url = re.search(r"https://[\w-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if url or tunnel.poll() is not None:
        break
    time.sleep(1)
if not url:
    raise RuntimeError("no public link came up:\n" + open("/content/tunnel.log").read()[-3000:])
print("open:", url.group(0))
print("Anyone with this link can use this ComfyUI, so keep it to yourself. It works while this runtime is connected.")
print("Logs: /content/comfy.log and /content/tunnel.log")
